## Inicializar los paquetes de Google Earth Engine (GEE)

Iniciando sesión con la cuenta previamente creada y obteniendo el código de permiso para el uso del API

In [1]:
import ee
import geemap
from ipyleaflet import GeoJSON

ee.Authenticate()
ee.Initialize()

Enter verification code:  4/1AWtgzh73RrN59s6jM3zwsIqGwTMeklR4FDsHjB_oEPI5tdiSEHmsS9mvJHc



Successfully saved authorization token.


## Obtener el polígono geojson

Teniendo en cuenta el área de interés a 20km a la redonda de Bogotá se establece un rectángulo que va a contener los datos satelitales

In [2]:
region = ee.Geometry.Rectangle(-74.638, 3.571, -73.710, 5.057)

map_data = geemap.Map()
map_data.setCenter(-74.2, 4.3, 9)

print(region)
print('----->Type: '+str(type(region)))

map_data.addLayer(region, {}, 'Región de interés', 1, 0.5)
map_data

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -74.638,
              5.057
            ],
            [
              -74.638,
              3.571
            ],
            [
              -73.71,
              3.571
            ],
            [
              -73.71,
              5.057
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})
----->Type: <class 'ee.geometry.Geometry'>


Map(center=[4.3, -74.2], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=…

## Conseguir los archivos .shp del mapa de Bogotá

Estos archivos fueron cargados previamente en la nube de GEE para su posterior tratamiento

In [3]:
bogota_localidades = ee.FeatureCollection('projects/ee-seforeros/assets/2022_localidades')
bogota_urbano = ee.FeatureCollection('projects/ee-seforeros/assets/2022_perimetro_urbano')
bogota_protegidas = ee.FeatureCollection('projects/ee-seforeros/assets/2022_areas_protegidas_sinap')
bogota_rural = ee.FeatureCollection('projects/ee-seforeros/assets/2022_territorio_rural')
estaciones = ee.FeatureCollection('projects/ee-seforeros/assets/CNE_IDEAM').filterBounds(region)

map_IDEAM = geemap.Map()
map_IDEAM.setCenter(-74.2, 4.3, 9)

#map.addLayer(bogota_localidades,{'color':"black"},'Localidades 2022',1)
map_IDEAM.addLayer (bogota_rural, {'color':"blue"} , "Territorio Rural",1,0.5)
map_IDEAM.addLayer(bogota_urbano,{'color':"black"},'Perímetro urbano 2022', 1, 0.5)
map_IDEAM.addLayer(bogota_protegidas,{'color':"red"},'Áreas protegidas SINAP 2022',1, 0.5)
map_IDEAM.addLayer(estaciones,{'color':"green"},'CNE IDEAM',1)

legend_dict = {
#    'Localidades 2022': '000000',
    'Territorio Rural': '0000FF',
    'Perímetro urbano 2022': '000000',
    'Áreas protegidas SINAP 2022': 'FF0000',
    'CNE IDEAM': '00FF00'
}

map_IDEAM.add_legend(legend_tittle = "Áreas de interés", legend_dict = legend_dict)
map_IDEAM

Map(center=[4.3, -74.2], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=…

**Example of the code for plot stellital data:**

    image_NAME = (ee.ImageCollection('extendsion of the data').
                  select('parameter').
                  filterDate('start-date', 'end-date'))

    band_viz_NAME = {
      'min': minimun,
      'max': maximum,
      'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']
    }

    mosaic = image_NAME.mosaic()
    clipped = mosaic.clip(region)

    map_data.setCenter(-74.161, 4.379, 9)
    map_data.addLayer(clipped, band_viz_NAME, 'text')
    map_data

# Datos del catálogo de GEE

## Fuente 1: Radiación solar
Variable: Radiación solar superficial neta \[J/m2\]  
Resolución: 11132m
Datos: [ERA5-Land Monthly Aggregated - ECMWF Climate Reanalysis](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR)

In [37]:
map_data.clear()
map_data = geemap.Map()

#Variables a considerar
source = "ECMWF/ERA5_LAND/MONTHLY_AGGR"
variable = 'Radiación solar superficial neta mensual'
band = 'surface_net_solar_radiation_sum'
units = ' [ J/m2 ]'
max_value = 550e6
min_value = 400e6
start = '2018-03-01'
end = '2018-04-01'

#Descarga de datos
image_region = ee.ImageCollection(source).filterBounds(region)
image_time = image_region.select(band).filterDate(start, end)
image_final = image_time.map(lambda image: image.clip(region))

#Banda de datos
band_viz = {
  'min': min_value,
  'max': max_value,
  'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}
map_data.setCenter(-74.2, 4.3, 9)
map_data.addLayer(image_final, band_viz, variable, 1, 0.8)

#Paleta de colores
colors = band_viz['palette']
vmin = band_viz['min']
vmax = band_viz['max']

map_data.add_colorbar(band_viz, label= variable + units)
map_data.addLayerControl()

#Capas de areas de interes
map_data.addLayer (bogota_rural, {'color':"0000FF"} , "Territorio Rural",1)
map_data.addLayer(bogota_protegidas,{'color':"FF0000"},'Áreas protegidas SINAP 2022',1)
map_data.addLayer(bogota_urbano,{'color':"000000"},'Perímetro urbano 2022', 1)

legend_dict_data = {
    'Territorio Rural': '0000FF',
    'Áreas protegidas SINAP 2022': 'FF0000',
    'Perímetro urbano 2022': '000000'
}
map_data.add_legend(legend_tittle = "Áreas de interés", legend_dict = legend_dict_data)

map_data

Map(center=[4.3, -74.2], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=…

## Fuente 1.2: Velocidad del viento
Variable: Velocidad del viento componente x \[m/s\]  
Resolución: 11132m
Datos: [ERA5-Land Monthly Aggregated - ECMWF Climate Reanalysis](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR)

In [45]:
map_data.clear()
map_data = geemap.Map()

#Variables a considerar
source = "ECMWF/ERA5_LAND/MONTHLY_AGGR"
variable = 'Componente x de velocidad'
band = 'u_component_of_wind_10m'
units = ' [ m/s ]'
max_value = -2
min_value = 2
start = '2018-03-01'
end = '2018-04-01'

#Descarga de datos
image_region = ee.ImageCollection(source).filterBounds(region)
image_time = image_region.select(band).filterDate(start, end)
image_final = image_time.map(lambda image: image.clip(region))

#Banda de datos
band_viz = {
  'min': min_value,
  'max': max_value,
  'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}
map_data.setCenter(-74.2, 4.3, 9)
map_data.addLayer(image_final, band_viz, variable, 1, 0.8)

#Paleta de colores
colors = band_viz['palette']
vmin = band_viz['min']
vmax = band_viz['max']

map_data.add_colorbar(band_viz, label= variable + units)
map_data.addLayerControl()

#Capas de areas de interes
map_data.addLayer (bogota_rural, {'color':"0000FF"} , "Territorio Rural",1)
map_data.addLayer(bogota_protegidas,{'color':"FF0000"},'Áreas protegidas SINAP 2022',1)
map_data.addLayer(bogota_urbano,{'color':"000000"},'Perímetro urbano 2022', 1)

legend_dict_data = {
    'Territorio Rural': '0000FF',
    'Áreas protegidas SINAP 2022': 'FF0000',
    'Perímetro urbano 2022': '000000'
}
map_data.add_legend(legend_tittle = "Áreas de interés", legend_dict = legend_dict_data)

map_data

Map(center=[4.3, -74.2], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=…

## Fuente 2: Velocidad del viento
Variable: Velocidad del viento medida a 10m de altura \[m/s\] (factor de escala 0.01)  
Resolución: 4638.3 m  
Datos: [TerraClimate: Monthly Climate and Climatic Water Balance for Global Terrestrial Surfaces, University of Idaho](https://developers.google.com/earth-engine/datasets/catalog/IDAHO_EPSCOR_TERRACLIMATE#bands)

In [47]:
map_data.clear()
map_data = geemap.Map()

#Variables a considerar
source = "IDAHO_EPSCOR/TERRACLIMATE"
variable = 'Velocidad total promedio mensual a 10m'
band = 'vs'
units = ' [ m/s ]'
max_value = 20
min_value = 100
start = '2018-03-01'
end = '2018-04-01'

#Descarga de datos
image_region = ee.ImageCollection(source).filterBounds(region)
image_time = image_region.select(band).filterDate(start, end)
image_final = image_time.map(lambda image: image.clip(region))

#Banda de datos
band_viz = {
  'min': min_value,
  'max': max_value,
  'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}
map_data.setCenter(-74.2, 4.3, 9)
map_data.addLayer(image_final, band_viz, variable, 1, 0.8)

#Paleta de colores
colors = band_viz['palette']
vmin = band_viz['min']
vmax = band_viz['max']

map_data.add_colorbar(band_viz, label= variable + units)
map_data.addLayerControl()

#Capas de areas de interes
map_data.addLayer (bogota_rural, {'color':"0000FF"} , "Territorio Rural",1)
map_data.addLayer(bogota_protegidas,{'color':"FF0000"},'Áreas protegidas SINAP 2022',1)
map_data.addLayer(bogota_urbano,{'color':"000000"},'Perímetro urbano 2022', 1)

legend_dict_data = {
    'Territorio Rural': '0000FF',
    'Áreas protegidas SINAP 2022': 'FF0000',
    'Perímetro urbano 2022': '000000'
}
map_data.add_legend(legend_tittle = "Áreas de interés", legend_dict = legend_dict_data)

map_data

Map(center=[4.3, -74.2], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=…